# A simple Polars Backtest / Replay formulation

We approximate execution on **configurable completed bars, default 1 second**.
All ticker calculations are native Polars expressions. One Python loop advances
time. The goal is to see the state and action formulation, not recreate a broker.

**Model:** decide at bar close -> fill at next bar open -> update positions/cash.
Fills are complete, with no spread, fees, partial fills, order book or reservations.
Stops/targets are close-based signals, also filled at the next open; they are not
intrabar stop orders and gaps can produce worse exit prices. Competing entries
share available cash proportionally, with quantities rounded down at fill time.
Sales are booked before purchases in the same opening batch. Residual cash stays
unused. There is one account, one long position per ticker, and no leverage.

Columns replace mutable ticker objects:
* market `[N, fields]`: open/close and a completed-bar entry signal;
* state `[N, fields]`: held quantity, entry price, stop/target and pending action;
* account `[1, fields]`: cash and cumulative realized P&L;
* actions `[N, fields]`: ENTER / EXIT / HOLD plus a requested quantity.

This is a research approximation, not equivalence to a released strategy or broker.


In [ ]:
import polars as pl

pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_cols(12)
RESOLUTION_SECONDS = 1  # Change to 2, 3, 4 or 6; this dummy source lasts 12 seconds.
parameters = pl.DataFrame({
    "entry_shares": [10], "stop_fraction": [0.05], "target_fraction": [0.10],
})
initial_account = pl.DataFrame({"cash": [600.0], "realized_pnl": [0.0]})
initial_state = pl.DataFrame({
    "ticker": ["AAA", "BBB", "CCC"], "quantity": [4, 0, 0],
    "entry_price": [100.0, 0.0, 0.0], "stop": [95.0, 0.0, 0.0],
    "target": [110.0, 0.0, 0.0], "pending_action": ["HOLD", "HOLD", "HOLD"],
    "requested_quantity": [0, 0, 0],
})

# Synthetic 1s source [12*N, fields]. Prices are dollars; floats keep this example
# readable. Real monetary precision would be an explicit separate model choice.
source = (
    pl.DataFrame({"second": pl.int_range(0, 12, eager=True)})
    .join(pl.DataFrame({"ticker": ["AAA", "BBB", "CCC"]}), how="cross")
    .with_columns(
        pl.when(pl.col("ticker") == "AAA")
          .then(pl.when(pl.col("second") < 2).then(100.0).otherwise(94.0))
          .when(pl.col("ticker") == "BBB")
          .then(pl.when(pl.col("second") < 4).then(50.0).otherwise(46.0))
          .otherwise(pl.when(pl.col("second") < 3).then(40.0).otherwise(46.0))
          .alias("close"),
        ((pl.col("second") == 0) & pl.col("ticker").is_in(["BBB", "CCC"]))
          .alias("entry_signal"),
    ).sort(["ticker", "second"])
    .with_columns(pl.col("close").shift(1).over("ticker")
                  .fill_null(pl.col("close")).alias("open"))
)

def make_bars(source, resolution_seconds):
    # Whole, aligned bars only. Do not silently omit a partial last bar.
    assert type(resolution_seconds) is int and resolution_seconds > 0
    assert source.select(pl.struct("ticker", "second").n_unique()).item() == source.height
    bars = (
        source.sort(["ticker", "second"])
        .with_columns((pl.col("second") // resolution_seconds).alias("bar"))
        .group_by(["bar", "ticker"]).agg(
            pl.col("open").first(), pl.col("close").last(),
            pl.col("entry_signal").any(), pl.len().alias("source_rows"),
            pl.col("second").min().alias("start_second"),
            (pl.col("second").max() + 1).alias("end_second"),
            pl.col("second").n_unique().alias("unique_seconds"),
        ).sort(["bar", "ticker"])
    )
    assert bars.select(((pl.col("source_rows") == resolution_seconds)
                        & (pl.col("unique_seconds") == resolution_seconds)
                        & (pl.col("start_second") == pl.col("bar") * resolution_seconds)
                        & (pl.col("end_second") == (pl.col("bar") + 1) * resolution_seconds))
                       .all()).item(), "Source must cover complete contiguous bars"
    return bars.drop("source_rows", "unique_seconds")

bars = make_bars(source, RESOLUTION_SECONDS)
print(bars)


## 1. Approximate execution: yesterday's action, today's open

Pending EXIT sells the entire holding. Pending ENTER requests shares; allocation
uses only this bar's opening prices and cash after sales. Today's close is not
used for fills. This changes share allocation when the next open gaps, rather
than borrowing cash or pretending the old close was executable.


In [ ]:
def fill_at_open(state, account, current_bar, parameters):
    rows = state.join(current_bar, on="ticker", validate="1:1").join(parameters, how="cross")
    rows = rows.with_columns(
        pl.when(pl.col("pending_action") == "EXIT").then(pl.col("quantity"))
          .otherwise(0).alias("sell_quantity"),
        pl.when((pl.col("pending_action") == "ENTER") & (pl.col("quantity") == 0))
          .then(pl.col("requested_quantity")).otherwise(0).alias("buy_request"),
    ).with_columns(
        (pl.col("sell_quantity") * pl.col("open")).alias("sale_cash"),
        (pl.col("sell_quantity") * (pl.col("open") - pl.col("entry_price")))
          .alias("realized_delta"),
        (pl.col("buy_request") * pl.col("open")).alias("requested_cost"),
    )
    # Scalar aggregations [1, fields], broadcast back to all N ticker requests.
    budget = account.join(rows.select(
        pl.col("sale_cash").sum(), pl.col("realized_delta").sum(),
        pl.col("requested_cost").sum().alias("total_requested_cost")), how="cross"
    ).with_columns((pl.col("cash") + pl.col("sale_cash")).alias("available_cash"))
    rows = rows.join(budget, how="cross").with_columns(
        pl.when(pl.col("total_requested_cost") > 0)
          .then(pl.min_horizontal(pl.lit(1.0),
                pl.col("available_cash") / pl.col("total_requested_cost")))
          .otherwise(0.0).alias("allocation_scale"),
    ).with_columns(
        (pl.col("buy_request") * pl.col("allocation_scale")).floor()
          .cast(pl.Int64).alias("buy_quantity"),
    ).with_columns(
        (pl.col("quantity") - pl.col("sell_quantity") + pl.col("buy_quantity"))
          .alias("next_quantity"),
        (pl.col("sale_cash") - pl.col("buy_quantity") * pl.col("open"))
          .alias("cash_delta"),
    )
    # Fill evidence is separate from the action that requested it.
    fills = rows.select("bar", "ticker", "start_second", "open", "pending_action",
                        "buy_request", "allocation_scale", "buy_quantity", "sell_quantity", "cash_delta")
    next_account = account.join(rows.select(
        pl.col("cash_delta").sum(), pl.col("realized_delta").sum()), how="cross"
    ).select((pl.col("cash") + pl.col("cash_delta")).alias("cash"),
             (pl.col("realized_pnl") + pl.col("realized_delta")).alias("realized_pnl"))
    # A new entry resets protection using its actual approximate opening fill.
    rows = rows.with_columns(
        pl.when(pl.col("buy_quantity") > 0).then(pl.col("open"))
          .when(pl.col("next_quantity") > 0).then(pl.col("entry_price"))
          .otherwise(0.0).alias("next_entry_price"),
        pl.when(pl.col("buy_quantity") > 0)
          .then(pl.col("open") * (1 - pl.col("stop_fraction")))
          .when(pl.col("next_quantity") > 0).then(pl.col("stop"))
          .otherwise(0.0).alias("next_stop"),
        pl.when(pl.col("buy_quantity") > 0)
          .then(pl.col("open") * (1 + pl.col("target_fraction")))
          .when(pl.col("next_quantity") > 0).then(pl.col("target"))
          .otherwise(0.0).alias("next_target"),
    )
    return rows, next_account, fills


## 2. Actions at completed close, then commit the next state

EXIT has priority when a held ticker closes through its stop or target. Otherwise
a flat ticker with a signal proposes ENTER; everything else proposes HOLD.
There are no orders to reconcile: a pending action is consumed once at the next
open, including an entry allocated zero shares. The final bar's pending actions
remain visible; no extra fill or end-of-run liquidation is fabricated.


In [ ]:
def decide_at_close(rows, account):
    decisions = rows.with_columns(
        pl.when((pl.col("next_quantity") > 0)
                & ((pl.col("close") <= pl.col("next_stop"))
                   | (pl.col("close") >= pl.col("next_target"))))
          .then(pl.lit("EXIT"))
          .when((pl.col("next_quantity") == 0) & pl.col("entry_signal"))
          .then(pl.lit("ENTER"))
          .otherwise(pl.lit("HOLD")).alias("action"),
    ).with_columns(
        pl.when(pl.col("action") == "ENTER").then(pl.col("entry_shares"))
          .when(pl.col("action") == "EXIT").then(pl.col("next_quantity"))
          .otherwise(0).alias("action_quantity"),
    )
    state = decisions.select(
        "ticker", pl.col("next_quantity").alias("quantity"),
        pl.col("next_entry_price").alias("entry_price"),
        pl.col("next_stop").alias("stop"), pl.col("next_target").alias("target"),
        pl.col("action").alias("pending_action"),
        pl.col("action_quantity").alias("requested_quantity"),
    ).sort("ticker")
    actions = decisions.select("bar", "ticker", "end_second", "close", "next_quantity",
                               "next_stop", "next_target", "action", "action_quantity")
    account_snapshot = account.join(decisions.select(
        (pl.col("next_quantity") * pl.col("close")).sum().alias("market_value"),
        (pl.col("next_quantity") * (pl.col("close") - pl.col("next_entry_price")))
          .sum().alias("unrealized_pnl")), how="cross"
    ).with_columns((pl.col("cash") + pl.col("market_value")).alias("equity"))
    return state, actions, account_snapshot

def run_replay(bars):
    state, account = initial_state, initial_account
    fill_batches, action_batches, state_batches, account_batches = [], [], [], []
    # The only Python data-processing loop is over TIME; no ticker/order row loop.
    # Replay can pace this loop; Backtest can run it as fast as possible.
    for current_bar in bars.sort(["bar", "ticker"]).partition_by("bar", maintain_order=True):
        assert current_bar.select(pl.col("ticker").n_unique()).item() == state.height
        assert state.join(current_bar, on="ticker", how="anti").is_empty()
        assert current_bar.select(((pl.col("open") > 0) & (pl.col("close") > 0)
                                  & pl.col("open").is_finite() & pl.col("close").is_finite()).all()).item()
        rows, account, fills = fill_at_open(state, account, current_bar, parameters)
        state, actions, snapshot = decide_at_close(rows, account)
        assert account.select((pl.col("cash") >= -1e-9).all()).item()
        clock = current_bar.select(pl.col("bar").first())
        fill_batches.append(fills)
        action_batches.append(actions)
        state_batches.append(state.join(clock, how="cross"))
        account_batches.append(snapshot.join(clock, how="cross"))
    return {"fills": pl.concat(fill_batches), "actions": pl.concat(action_batches),
            "states": pl.concat(state_batches), "accounts": pl.concat(account_batches)}

result = run_replay(bars)
print("Close-based action proposals")
print(result["actions"].filter(pl.col("action") != "HOLD"))
print("Approximate next-open fills")
print(result["fills"].filter((pl.col("buy_quantity") + pl.col("sell_quantity")) > 0))
print("Account snapshots")
print(result["accounts"])


## Small checks and resolution comparison

The same transition expressions run at 1s or 3s. Different resolution can produce
different entry prices and decisions: it changes bar availability and the next
executable opening, not just replay speed. Future closes cannot alter past state
or same-bar opening fills. Changing resolution must use bars built at that
resolution; relabeling a timestamp is insufficient.


In [ ]:
cash_check = result["accounts"].join(
    result["fills"].group_by("bar").agg(pl.col("cash_delta").sum()).sort("bar")
      .with_columns(pl.col("cash_delta").cum_sum().alias("cumulative_cash_delta")), on="bar")
assert cash_check.select(((pl.col("cash") - 600 - pl.col("cumulative_cash_delta")).abs() < 1e-9).all()).item()
assert result["accounts"].select(
    ((pl.col("equity") - 1_000 - pl.col("realized_pnl") - pl.col("unrealized_pnl"))
     .abs() < 1e-9).all()).item()
assert result["fills"].filter(pl.col("bar") == 0).select(
    ((pl.col("buy_quantity") + pl.col("sell_quantity")) == 0).all()).item()

# Alter a completed close; this can change that close's action, not its open fill.
changed_close = bars.with_columns(
    pl.when(pl.col("bar") == 1).then(pl.col("close") * 0.5)
      .otherwise(pl.col("close")).alias("close"))
changed_result = run_replay(changed_close)
assert result["fills"].filter(pl.col("bar") <= 1).equals(changed_result["fills"].filter(pl.col("bar") <= 1))
assert result["states"].filter(pl.col("bar") == 0).equals(changed_result["states"].filter(pl.col("bar") == 0))

one_second = run_replay(make_bars(source, 1))
three_seconds = run_replay(make_bars(source, 3))
comparison = pl.concat([
    one_second["accounts"].tail(1).with_columns(pl.lit(1).alias("resolution_seconds")),
    three_seconds["accounts"].tail(1).with_columns(pl.lit(3).alias("resolution_seconds")),
]).select("resolution_seconds", "cash", "market_value", "equity", "realized_pnl")
assert one_second["fills"].filter((pl.col("bar") == 1) & pl.col("ticker").is_in(["BBB", "CCC"])).select(
    (pl.col("buy_quantity") == 6).all()).item()
print(comparison)
print("PASS: accounting identities, next-bar execution, close/open causality, and 1s/3s runs.")
